# Knee MRI Classifier — MedGemma Fine-Tuning for ACL Tear Detection

**Course:** COMP 6630 Machine Learning, Spring 2026  
**Team:** Gabriella Hawkes, Elizabeth Casey, Maddie Larkin  
**Dataset:** MRNet (Stanford) — knee MRI exams labeled for ACL tear  
**Model:** google/medgemma-4b-it (vision-language model, pretrained on medical imagery)

This notebook establishes a zero-shot baseline and fine-tunes MedGemma on the MRNet dataset for binary ACL tear classification. Follows the MedGemma fine-tuning tutorial: https://github.com/google-health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb

### GenAI Disclosure
Used Claude and Gemini to: debug data pipeline issues (parquet loading, HuggingFace fixed-shape decoder workarounds, PyArrow→HF Dataset conversion), diagnose out-of-memory errors, draft model-loading boilerplate, and write the zero-shot baseline evaluation code.


## 1. Setup

In [ ]:
!pip install -q bitsandbytes accelerate peft trl

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from huggingface_hub import login, hf_hub_download, snapshot_download
from transformers import pipeline
from torchvision import transforms
from torch import Tensor
from sklearn.preprocessing import normalize
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import glob


### HuggingFace login

**Before running this cell:**
1. Accept the MedGemma license at https://huggingface.co/google/medgemma-4b-it (you must be logged in to HuggingFace; requests are processed immediately).
2. Generate a **Read** access token at https://huggingface.co/settings/tokens (a fine-grained token will not work with gated repos unless specifically configured).
3. Paste the token when prompted below.


In [ ]:
login()

## 2. Download and inspect the dataset

In [ ]:
def download_one_example(repo_id, filepath):
    """Download one parquet shard from HF and return it as an HF Dataset."""
    filepath1 = hf_hub_download(
        repo_id=repo_id,
        filename=filepath,
        repo_type="dataset",
    )
    ds = pq.read_table(filepath1)
    return Dataset(ds)


In [ ]:
# Quick peek at a single parquet shard
example = download_one_example(
    "AUMLProject/mrnet-acl",
    "data/train-00000-of-00029.parquet",
)
print(example)

### Load the full dataset

We use MRNet's **official validation set** as our test set (instead of a random split). This keeps our numbers comparable to published MRNet benchmark results and avoids patient-level leakage that a random split might introduce.

The `n_files` argument lets us load a subset of training shards for faster iteration. Set `n_files=None` to use all 29 shards (full training set, ~6.5 GB).


In [ ]:
def download_train_test_datasets(seed=42, n_files=None):
    """Load MRNet ACL dataset from HuggingFace with memory-mapping."""
    if n_files is None:
        train_ds = load_dataset("AUMLProject/mrnet-acl", split="train")
    else:
        data_files = [f"data/train-{i:05d}-of-00029.parquet" for i in range(n_files)]
        train_ds = load_dataset(
            "AUMLProject/mrnet-acl",
            data_files={"train": data_files},
            split="train",
            verification_mode="no_checks",
        )

    test_ds = load_dataset("AUMLProject/mrnet-acl", split="validation")
    ds = DatasetDict({"train": train_ds, "test": test_ds})

    print(f"Train size: {len(ds['train'])}")
    print(f"Test size:  {len(ds['test'])}")
    print(f"Train label dist: {pd.Series(ds['train']['label']).value_counts().to_dict()}")
    print(f"Test label dist:  {pd.Series(ds['test']['label']).value_counts().to_dict()}")
    return ds


In [ ]:
# For the graded submission we use n_files=None (full training set).
# During development we used n_files=3 (~10%) for faster iteration.
ds = download_train_test_datasets(n_files=3)

In [ ]:
print(ds)  # Each view (sagittal/coronal/axial) is a 3D MRI stack

### Inspect a sample — confirming variable slice counts

The dataset schema declares each view as a fixed shape `(32, 224, 224)`, but actual exams have variable slice counts (typically 25–40). This mismatch means `sample[view]` via the standard HuggingFace API throws a reshape error. We bypass it by reading raw data directly from the underlying PyArrow table.

In [ ]:
test_table = ds["test"].data.table
row_idx = 0
for view in ["sagittal", "coronal", "axial"]:
    raw = test_table.column(view)[row_idx].as_py()
    arr = np.array(raw)
    print(f"{view}: shape {arr.shape}, dtype {arr.dtype}")
label = test_table.column("label")[row_idx].as_py()
print(f"label: {label}")

## 3. Zero-shot baseline (MedGemma with no fine-tuning)

Before fine-tuning, we measure how well MedGemma classifies ACL tears out-of-the-box. This is the "before" number that the fine-tuned model must beat.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from transformers import pipeline
from PIL import Image
import bitsandbytes as bnb
import accelerate


In [ ]:
# We use the same base model for the baseline as for fine-tuning, so the two
# sets of metrics are directly comparable.
model_id = "google/medgemma-4b-it"

In [ ]:
# Load with 4-bit quantization so the model fits on a T4 GPU (~8 GB download,
# ~3-5 min on first run).
model_kwargs = dict(
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
)
pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)

### Helper functions

In [ ]:
def get_sample(ds_split, idx):
    """Read one sample using the PyArrow table directly — bypasses HF's
    fixed-shape decoder so variable slice counts work correctly.

    Returns a dict with 'sagittal', 'coronal', 'axial' as numpy arrays of
    shape (n_slices, H, W) and 'label' as an int.
    """
    table = ds_split.data.table
    return {
        "sagittal": np.array(table.column("sagittal")[idx].as_py()),
        "coronal":  np.array(table.column("coronal")[idx].as_py()),
        "axial":    np.array(table.column("axial")[idx].as_py()),
        "label":    int(table.column("label")[idx].as_py()),
    }


def slice_to_pil(mri_stack, slice_idx=None):
    """Convert one slice of a 3D MRI stack to a PIL image, rescaled to [0, 255]."""
    n_slices = mri_stack.shape[0]
    if slice_idx is None:
        slice_idx = n_slices // 2  # middle slice by default
    if not (0 <= slice_idx < n_slices):
        raise ValueError(f"slice_idx {slice_idx} out of [0, {n_slices})")

    sl = mri_stack[slice_idx].astype(np.float32)
    sl_min, sl_max = sl.min(), sl.max()
    if sl_max - sl_min < 1e-8:
        rescaled = np.zeros_like(sl, dtype=np.uint8)
    else:
        rescaled = (255.0 * (sl - sl_min) / (sl_max - sl_min)).astype(np.uint8)
    return Image.fromarray(rescaled)


def parse_yes_no(response_text):
    """Parse MedGemma's free-text response into a 0/1 prediction.
    Returns None if the response cannot be interpreted.
    """
    if not response_text:
        return None
    txt = response_text.strip().lower()
    words = txt.split()
    if not words:
        return None
    first = words[0].strip(".,:;!?'\"")
    if first == "yes":
        return 1
    if first == "no":
        return 0
    if "yes" in txt and "no" not in txt:
        return 1
    if "no" in txt and "yes" not in txt:
        return 0
    return None


### Inference function — no label leakage

`predict_acl` builds a user-only chat message (no assistant turn). The model has to generate its answer from scratch — we're measuring its actual prediction ability, not its ability to echo a label we accidentally included.


In [ ]:
def predict_acl(sample, pipe, view="sagittal", num_center_slices=1):
    """Zero-shot (or fine-tuned) ACL tear prediction on one MRNet sample.

    Args:
        sample: dict from get_sample()
        pipe: image-text-to-text pipeline (baseline or fine-tuned)
        view: 'sagittal', 'coronal', or 'axial'
        num_center_slices: how many center slices to feed the model
    """
    stack = sample[view]
    n_slices = stack.shape[0]
    mid = n_slices // 2
    num_each_side = num_center_slices // 2
    is_odd = 1 if num_center_slices % 2 != 0 else 0
    start = max(0, mid - num_each_side)
    end = min(n_slices, mid + num_each_side + is_odd)
    images = [slice_to_pil(stack, i) for i in range(start, end)]

    prompt = (
        "You are looking at a knee MRI slice. Does this image "
        "show evidence of an ACL (Anterior Cruciate Ligament) "
        "tear? Answer with a single word: 'Yes' or 'No'."
    )
    user_content = [{"type": "text", "text": prompt}]
    for img in images:
        user_content.append({"type": "image", "image": img})
    messages = [{"role": "user", "content": user_content}]

    with torch.no_grad():
        output = pipe(text=messages, max_new_tokens=10, do_sample=False)
        response = output[0]["generated_text"][-1]["content"]

    return {
        "response": response,
        "prediction": parse_yes_no(response),
        "label": sample["label"],
        "view": view,
    }


### Training-format helper (used in the fine-tuning section)

`reformat_sample` DOES include the ground-truth answer as the assistant turn — that's how SFTTrainer learns what the correct response is. Do not use this for inference (use `predict_acl` instead).

In [ ]:
def reformat_sample(stack, label, num_center_slices=1):
    """Build one chat-format training example for SFTTrainer.

    Includes the ground-truth label in the assistant turn so the model learns
    from it. DO NOT use for inference — that would leak the label.
    """
    n_slices = stack.shape[0]
    mid = n_slices // 2
    num_each_side = num_center_slices // 2
    is_odd = 1 if num_center_slices % 2 != 0 else 0
    start = max(0, mid - num_each_side)
    end = min(n_slices, mid + num_each_side + is_odd)
    images = [slice_to_pil(stack, i) for i in range(start, end)]

    prompt = (
        "You are looking at a knee MRI slice. Does this image "
        "show evidence of an ACL (Anterior Cruciate Ligament) "
        "tear? Answer with a single word: 'Yes' or 'No'."
    )
    user_content = [{"type": "text", "text": prompt}]
    for img in images:
        user_content.append({"type": "image", "image": img})
    answer = "Yes" if label == 1 else "No"

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]},
    ]
    return {"messages": messages, "image": images[0]}


def reformat_ds_at_index(ds_split, idx, view="sagittal", num_center_slices=1):
    table = ds_split.data.table
    stack = np.array(table.column(view)[idx].as_py())
    label = int(table.column("label")[idx].as_py())
    return reformat_sample(stack, label, num_center_slices)


def build_finetuning_dataset(ds_split, view="sagittal", num_center_slices=1):
    """Build a new HF Dataset with 'messages' and 'image' columns for SFTTrainer.
    Uses raw PyArrow access so the fixed-shape decoder never fires.
    """
    from datasets import Dataset as HFDataset
    records = []
    for idx in range(len(ds_split)):
        records.append(reformat_ds_at_index(ds_split, idx, view, num_center_slices))
    return HFDataset.from_list(records)


### Sanity check: one sample before full eval

In [ ]:
sample = get_sample(ds["test"], 0)
result = predict_acl(sample, pipe, view="sagittal")
print(f"Label:        {result['label']}")
print(f"Prediction:   {result['prediction']}")
print(f"Raw response: {result['response']!r}")

### Baseline evaluation

We evaluate on the full test set (n=120) using the sagittal view, which the MRNet paper identifies as the most diagnostic for ACL tears. Metrics include accuracy, precision, recall, F1, and a confusion matrix — all relevant for this imbalanced binary classification task.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from tqdm.auto import tqdm


def evaluate(pipe, test_ds, n_samples=None, view="sagittal", seed=42,
             verbose=True, label="MedGemma"):
    """Run pipe over test samples and report classification metrics.

    Args:
        pipe: image-text-to-text pipeline (zero-shot or fine-tuned)
        test_ds: HF Dataset with 'sagittal', 'coronal', 'axial', 'label'
        n_samples: if None, evaluate on the full test set; else random subset
        view: which MRI view to send to the model
        seed: for reproducibly sampling indices when n_samples is set
        label: human-readable label for the printed report header
    """
    if n_samples is None:
        indices = np.arange(len(test_ds))
    else:
        n = min(n_samples, len(test_ds))
        indices = np.random.RandomState(seed).choice(len(test_ds), size=n, replace=False)

    results = []
    for idx in tqdm(indices, desc=f"Evaluating {label} ({view})"):
        sample = get_sample(test_ds, int(idx))
        try:
            r = predict_acl(sample, pipe, view=view)
            r["idx"] = int(idx)
            results.append(r)
            if verbose:
                print(f"[{idx}] label={r['label']} pred={r['prediction']} | "
                      f"{r['response'][:60]}")
        except Exception as e:
            print(f"[{idx}] ERROR: {e}")
            results.append({
                "idx": int(idx), "prediction": None,
                "label": sample["label"], "response": f"ERROR: {e}",
                "view": view,
            })

    y_true, y_pred = [], []
    n_unparsed = 0
    for r in results:
        y_true.append(int(r["label"]))
        if r["prediction"] is None:
            n_unparsed += 1
            y_pred.append(1 - int(r["label"]))  # unparseable → counted wrong
        else:
            y_pred.append(int(r["prediction"]))

    metrics = {
        "n_evaluated": len(y_true),
        "n_unparseable": n_unparsed,
        "view": view,
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

    print("\n" + "=" * 60)
    print(f"{label} — view={view}, n={metrics['n_evaluated']}")
    print("=" * 60)
    print(f"Accuracy:  {metrics['accuracy']:.3f}")
    print(f"Precision: {metrics['precision']:.3f}")
    print(f"Recall:    {metrics['recall']:.3f}")
    print(f"F1:        {metrics['f1']:.3f}")
    print(f"Confusion: {metrics['confusion_matrix']}")
    print("  (rows = true [no-tear, tear], cols = predicted [no-tear, tear])")
    print(f"Unparseable responses: {metrics['n_unparseable']}")
    print()
    print(classification_report(
        y_true, y_pred, target_names=["no-tear", "tear"], zero_division=0
    ))
    return results, metrics


# Alias for backward compatibility with earlier versions of the notebook.
evaluate_baseline = evaluate

In [ ]:
# Zero-shot baseline on the full test set (n=120).
# For a quick check during development, pass n_samples=30.
baseline_results, baseline_metrics = evaluate(
    pipe, ds["test"],
    n_samples=None,       # full test set; use 30 for quick dev iteration
    view="sagittal",
    label="Zero-shot MedGemma",
    verbose=False,
)

In [ ]:
import json

with open("baseline_results_sagittal.json", "w") as f:
    json.dump({
        "model": model_id,
        "fine_tuned": False,
        "view": "sagittal",
        "n_samples": baseline_metrics["n_evaluated"],
        "metrics": baseline_metrics,
    }, f, indent=2)

print("Saved baseline_results_sagittal.json")
print(json.dumps(baseline_metrics, indent=2))

## 4. Fine-tuning MedGemma with LoRA

We fine-tune MedGemma using LoRA adapters on top of the 4-bit quantized base model. This is parameter-efficient and fits comfortably in a T4 GPU.

Following the MedGemma fine-tuning tutorial: https://github.com/google-health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb

In [ ]:
# Free up GPU memory from the zero-shot pipeline before loading the training model.
import gc
try:
    del pipe
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

### Build training-format datasets

In [ ]:
# Uses raw PyArrow access to bypass the fixed-shape decoder and .map() entirely.
train_ds = build_finetuning_dataset(ds["train"], view="sagittal", num_center_slices=1)
eval_ds  = build_finetuning_dataset(ds["test"],  view="sagittal", num_center_slices=1)

print(f"Train: {len(train_ds)} examples")
print(f"Eval:  {len(eval_ds)} examples")
print("\nFirst example keys:", list(train_ds[0].keys()))
print("Sample messages (first example):")
for msg in train_ds[0]["messages"]:
    print(f"  {msg['role']}: {[c.get('type') for c in msg['content']]}")

### Load the training model

In [ ]:
# Branch on GPU capability: bfloat16 on A100/H100, float16 on T4 and older.
if torch.cuda.get_device_capability()[0] < 8:
    model_kwargs = dict(
        attn_implementation="eager",
        torch_dtype=torch.float16,
        device_map="auto",
    )
else:
    model_kwargs = dict(
        attn_implementation="eager",
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=model_kwargs["torch_dtype"],
    bnb_4bit_quant_storage=model_kwargs["torch_dtype"],
)

model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"  # required for training

### LoRA configuration

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
)

### Collation function

In [ ]:
from typing import Any


def collate_fn(examples: list[dict[str, Any]]):
    """Batch collation matching MedGemma's expected input format.
    Masks pad tokens and image tokens from the loss computation.
    """
    texts = []
    images = []
    for example in examples:
        images.append([example["image"].convert("RGB")])
        texts.append(processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False,
        ).strip())

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100  # additional image-token variant
    batch["labels"] = labels
    return batch

### Training configuration and run

We train for a small number of epochs with a conservative learning rate suitable for LoRA. Batch size is small because each example contains a high-resolution image. Adjust `num_train_epochs` up if you have time and want to push accuracy further.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="medgemma-mrnet-sagittal",
    num_train_epochs=1,               # bump to 2-3 for final runs
    per_device_train_batch_size=2,    # small because images are large
    gradient_accumulation_steps=8,    # effective batch size 16
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    bf16=(torch.cuda.get_device_capability()[0] >= 8),
    fp16=(torch.cuda.get_device_capability()[0] < 8),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_kwargs={"skip_prepare_dataset": True},  # our dataset is pre-formatted
    remove_unused_columns=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate_fn,
    peft_config=peft_config,
    processing_class=processor,
)

In [ ]:
# Kick off training. Expect roughly 20-40 minutes on a T4 for 1 epoch with
# n_files=3 (~117 training examples). Scales roughly linearly with dataset size.
trainer.train()

In [ ]:
# Save the fine-tuned LoRA adapters so we can reload the model later.
trainer.save_model("medgemma-mrnet-sagittal/final")
print("Saved fine-tuned adapters to medgemma-mrnet-sagittal/final")

## 5. Evaluate the fine-tuned model

We rebuild a pipeline around the fine-tuned model (base + LoRA adapters merged) and re-run the same evaluation we used for the baseline. Identical data, identical metrics — so the before/after comparison is apples-to-apples.

In [ ]:
# Build an inference pipeline on the fine-tuned model.
# Merge the LoRA adapters into the base model's forward pass.
finetuned_pipe = pipeline(
    "image-text-to-text",
    model=trainer.model,
    tokenizer=processor.tokenizer,
    image_processor=processor.image_processor,
)

In [ ]:
# Sanity-check the fine-tuned model on one sample.
sample = get_sample(ds["test"], 0)
result = predict_acl(sample, finetuned_pipe, view="sagittal")
print(f"Label:        {result['label']}")
print(f"Prediction:   {result['prediction']}")
print(f"Raw response: {result['response']!r}")

In [ ]:
# Full evaluation of the fine-tuned model.
finetuned_results, finetuned_metrics = evaluate(
    finetuned_pipe, ds["test"],
    n_samples=None,
    view="sagittal",
    label="Fine-tuned MedGemma",
    verbose=False,
)

In [ ]:
with open("finetuned_results_sagittal.json", "w") as f:
    json.dump({
        "model": model_id,
        "fine_tuned": True,
        "view": "sagittal",
        "n_samples": finetuned_metrics["n_evaluated"],
        "metrics": finetuned_metrics,
    }, f, indent=2)

print("Saved finetuned_results_sagittal.json")

## 6. Before vs. After — comparison for the report

This cell prints a clean side-by-side of zero-shot and fine-tuned metrics. Paste this output directly into the final report's results section.

In [ ]:
def print_comparison(baseline, finetuned):
    rows = [
        ("Accuracy",  baseline["accuracy"],  finetuned["accuracy"]),
        ("Precision", baseline["precision"], finetuned["precision"]),
        ("Recall",    baseline["recall"],    finetuned["recall"]),
        ("F1",        baseline["f1"],        finetuned["f1"]),
    ]
    print("=" * 65)
    print(f"{'Metric':<12}{'Zero-shot':>18}{'Fine-tuned':>18}{'Delta':>15}")
    print("-" * 65)
    for name, b, f in rows:
        delta = f - b
        sign = "+" if delta >= 0 else ""
        print(f"{name:<12}{b:>18.3f}{f:>18.3f}{sign}{delta:>14.3f}")
    print("=" * 65)
    print(f"\nZero-shot confusion matrix:  {baseline['confusion_matrix']}")
    print(f"Fine-tuned confusion matrix: {finetuned['confusion_matrix']}")
    print("  (rows = true [no-tear, tear], cols = predicted [no-tear, tear])")


print_comparison(baseline_metrics, finetuned_metrics)

In [ ]:
# Save a combined comparison file for the report.
with open("comparison_sagittal.json", "w") as f:
    json.dump({
        "model": model_id,
        "view": "sagittal",
        "baseline": baseline_metrics,
        "finetuned": finetuned_metrics,
    }, f, indent=2)
print("Saved comparison_sagittal.json")